In [1]:
!pip -q install --upgrade huggingface_hub 
!apt -q install git -y
!pip -q install groq
!pip -q install python-dotenv

Reading package lists...
Building dependency tree...
Reading state information...
git is already the newest version (1:2.17.1-1ubuntu0.18).
0 upgraded, 0 newly installed, 0 to remove and 58 not upgraded.


In [2]:
from mario_gpt.prompter import Prompter
from mario_gpt import MarioDataset, MarioLM
from mario_gpt.prompt_adapter import PromptAdapter, PromptSplitter
from transformers import pipeline
from transformers import AutoTokenizer
from groq import Groq
from huggingface_hub import login
from dotenv import load_dotenv
import os
import json
from tqdm import tqdm

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# DEFAULT_MODEL = "llama-3.2-3b-preview"
DEFAULT_MODEL = "llama-3.1-8b-instant"

# llm_model = pipeline("text-generation", model="meta-llama/Llama-3.2-3B", device=0)
mario_lm = MarioLM()

Using shyamsn97/Mario-GPT2-700-context-length lm


/opt/conda/lib/python3.10/site-packages/transformers-4.46.3-py3.10.egg/transformers/models/auto/modeling_auto.py:1833: FutureWarning: The class `AutoModelWithLMHead` is deprecated and will be removed in a future version. Please use `AutoModelForCausalLM` for causal language models, `AutoModelForMaskedLM` for masked language models and `AutoModelForSeq2SeqLM` for encoder-decoder models.
  warnings.warn(


Using shyamsn97/Mario-GPT2-700-context-length tokenizer


In [3]:
# Initialize
# splitter = PromptSplitter(llm=llm_model)
splitter = PromptSplitter(llm=client, max_level_size=3, DEFAULT_MODEL=DEFAULT_MODEL)

# Test prompt with sequence
# prompt = "Make a level that starts with many goombas, then starts to introduce koopas, and finally ends with lots of coins and powerups"

# prompt = "Craft a level that is crazy right of the bat with a bunch of enemies, such as goombas and koopas. Then, add some powerups to help the player, but then end with with many blocks at high elevations to difficult platforming"

prompt = "Generate a level with many goombas, some turtles (koopas) that can be used for shell power-ups, some floating coins, and no special enemies."

In [4]:
sections = splitter.split_prompt(prompt, use_groq=True)
print("Split sections:")
for section in sections:
    print(f"- {section}")

# Then test full generation
# level = splitter.generate_levels(prompt, mario_lm)

Split sections:
- 1. "Generate a level with many goombas and no special enemies."
- 2. "Generate a level that includes some turtles (koopas) that can be used for shell power-ups and no special enemies."
- 3. "Generate a level that includes some floating coins and no special enemies."


In [5]:
# results = {
#     "original_prompt": prompt,
#     "splits": []
# }
# 
# for i in tqdm(range(50), desc="Generating splits"):
#     sections = splitter.split_prompt(prompt, use_groq=True)
#     results["splits"].append(sections)
# 
# with open('prompt_splits.json', 'w') as f:
#     json.dump(results, f, indent=2)